In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS houseprices.gold;

In [0]:
import gc
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType, StringType

# Carrega os dados perfeitamente limpos da Silver
df_silver = spark.table("houseprices.silver.silver_train")

coluna_target = "SalePrice"
colunas_ignoradas = ["Id", coluna_target]

# Mapeia quem é número e quem é texto
colunas_numericas = [
    f.name for f in df_silver.schema.fields 
    if isinstance(f.dataType, (DoubleType, IntegerType, LongType, FloatType)) 
    and f.name not in colunas_ignoradas
]

colunas_categoricas = [
    f.name for f in df_silver.schema.fields 
    if isinstance(f.dataType, StringType) 
    and f.name not in colunas_ignoradas
]

# Prepara os nomes das colunas futuras
colunas_indexed = [c + "_indexed" for c in colunas_categoricas]
colunas_encoded = [c + "_encoded" for c in colunas_categoricas]

coluna_exemplo = "MSZoning" # Focaremos nesta coluna para ver a transformação
print("Setup concluído. Pronto para iniciar o Pipeline.")

##  String_indexer: Transforma cada categoria em um inteiro

In [0]:

string_indexer = StringIndexer(
    inputCols=colunas_categoricas,
    outputCols=colunas_indexed,
    handleInvalid="keep"
)

# Treina e aplica APENAS o Indexer para visualizarmos
indexer_model = string_indexer.fit(df_silver)
df_etapa_a = indexer_model.transform(df_silver)

print("SAÍDA DA ETAPA A (As categorias de texto viram números Float):")
display(df_etapa_a.select("Id", coluna_exemplo, f"{coluna_exemplo}_indexed").dropDuplicates([coluna_exemplo]))

## 

## OneHotEncoder: Transforma cada categoria em uma representação em colunas

In [0]:
from pyspark.ml.functions import vector_to_array

one_hot_encoder = OneHotEncoder(
    inputCols=colunas_indexed,
    outputCols=colunas_encoded
)

encoder_model = one_hot_encoder.fit(df_etapa_a)
df_etapa_b = encoder_model.transform(df_etapa_a)

df_etapa_b = df_etapa_b.withColumn(
    f"{coluna_exemplo}_matriz_aberta", 
    vector_to_array(f"{coluna_exemplo}_encoded")
)

print("SAÍDA DA ETAPA B (Vetor Aberto: Mostrando todos os zeros e uns reais):")
display(df_etapa_b.select(
    "Id", 
    coluna_exemplo, 
    f"{coluna_exemplo}_indexed", 
    f"{coluna_exemplo}_encoded",    
    f"{coluna_exemplo}_matriz_aberta" 
).dropDuplicates([coluna_exemplo]))

## Vector Assembler: empacota todas as variáveis em uma única coluna de features

In [0]:
colunas_finais_features = colunas_numericas + colunas_encoded
vector_assembler = VectorAssembler(
    inputCols=colunas_finais_features,
    outputCol="features"
)

df_etapa_c = vector_assembler.transform(df_etapa_b)

print("SAÍDA DA ETAPA C (Como a IA enxerga o dado: Tudo compactado na super-coluna 'features'):")
display(df_etapa_c.select("Id", coluna_exemplo, "LotArea", f"{coluna_exemplo}_encoded", "features").dropDuplicates([coluna_exemplo]))

In [0]:
pipeline = Pipeline(stages=[string_indexer, one_hot_encoder, vector_assembler])
pipeline_model = pipeline.fit(df_silver)
df_gold = pipeline_model.transform(df_silver)


df_gold_final = df_gold.select("Id", "features", coluna_target)

df_gold_final.write.format("delta").mode("overwrite").saveAsTable("houseprices.gold.gold_train_ml")
print("Tabela Gold otimizada salva com sucesso!")

del indexer_model
del encoder_model
del pipeline_model  
gc.collect()        
print("Memória limpa e pronta para o treinamento da IA!")

In [0]:
df_gold = spark.table("houseprices.gold.gold_train_ml")

display(df_gold.limit(10))